## Retrieval-Augmented Generation (RAG)

Large Language Models (LLMs) generate responses primarily from knowledge acquired during training. Consequently, they do not automatically have access to specialised or private collections, such as a locally stored research corpus. **Retrieval-Augmented Generation (RAG)** addresses this limitation by combining an LLM with an external information retrieval system.[^1]

Instead of relying exclusively on the model's internal knowledge, a RAG system first **retrieves information relevant to the user's query** and then provides this information to the LLM as additional context. This is particularly useful when working with specialised corpora that were not part of the model's training data, or collections that are too large to fit into the model's context window.[^1]

### A simplified RAG pipeline can be represented as:

> **User question → retrieve relevant documents → add documents to the context → LLM → generated answer**

RAG does not normally retrain the language model on the external collection. Instead, the external data are made available to the model **at inference time**.[^1]





### Indexing the corpus

Before documents can be retrieved, the corpus must be prepared for efficient search. In a typical embedding-based RAG workflow, this consists of four main operations described by LangChain as **load, split, embed, and store**:[^1]

1. **Load** – import the source documents.
2. **Split** – divide long documents into smaller units or chunks where necessary.
3. **Embed** – convert each document into a numerical vector representing its semantic content.
4. **Store** – save the embeddings and associated documents in a vector store such as Chroma.

For a corpus in which each CSV row already represents a short document, additional splitting may not be necessary. Each row can instead become a single retrievable document.

The resulting workflow is approximately:

**CSV → Documents → Embedding model → Vector store**

At query time, the process is reversed into a retrieval pipeline:

**Question → Query embedding → Similarity search → Relevant documents → LLM → Answer**


### Embedding-based RAG

The most common RAG architecture uses **vector embeddings** for retrieval. An embedding model transforms a piece of text into a numerical vector. Texts that are semantically similar should occupy relatively similar positions in this vector space.

For example, a document discussing *negative representations of minority groups* might be retrieved for a query about *hostile portrayals of minorities*, even when the exact vocabulary differs.

The basic retrieval process is:

```text
document → embedding ─┐
document → embedding ─┤
document → embedding ─┤
                      ├→ similarity → most relevant documents
question → embedding ─┘
```

Vector databases such as Chroma store these representations and provide efficient similarity search.[^2]

Embedding-based retrieval is particularly effective when the principal goal is to answer questions such as:

> *Which documents are semantically relevant to this question?*

It is relatively simple to implement, works well with unstructured text, and can be combined with metadata filtering. For example, semantic retrieval can be restricted to documents from a particular **year, newspaper, topic, category, or sentiment class**.

Its principal limitation is that **semantic similarity is not the same as an explicit relationship**. Two documents may be related through a person, organisation, event, temporal sequence, or other relationship even when their textual embeddings are not particularly similar.



# Example:  What does "self-reflection" mean in the context of agentic AI?

## Task decomoposition:

<b> Goal → reason about what needs to be done → create subtasks → execute them → inspect results → revise the plan → continue </b>

### Installations (skip if not neccessary)

In [1]:
!pip install -U \
    langchain \
    langchain-openai \
    langchain-chroma \
    langchain-docling \
    langchain-community \
    langchain-text-splitters \
    chromadb \
    docling \
    beautifulsoup4

INFO: pip is looking at multiple versions of docling-slim[convert-core,feat-chunking,service-client] to determine which version is compatible with other requirements. This could take a while.
  Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl.metadata (62 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.9/732.9 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 567.0/567.0 kB 30.6 MB/s eta 0:00:00
Using cached numpy-2.2.6-cp312-cp312-macosx_14_0_arm64.whl (5.1 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.5.4
    Uninstalling langchain-core-1.5.4:
      Successfully uninstalled langchain-core-1.5.4
  Attempting uninstall: langchain-openai
    Found existing installation: langchain-openai 1.5.1
    Uninstalling langchain-openai-1.5.1:
      Successfully uninstal

In [2]:
import sys
!{sys.executable} -m pip install -U langchain-community

In [5]:
import sys
import numpy as np

print(sys.executable)
print(np.__version__)
print(np.__file__)

/opt/anaconda3/bin/python
2.2.6
/opt/anaconda3/lib/python3.12/site-packages/numpy/__init__.py


In [7]:
!pip install numpy==1.26.4

  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 5.0.0.93 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 7.35.1 which is incompatible.


### Main imports

In [13]:
%pip install --force-reinstall "numpy<2" "pandas>=2.2,<3"


  Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached pytz-2026.3.post1-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached numpy-1.26.4-cp312-cp312-macosx_11_0_arm64.whl (13.7 MB)
Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl (10.7 MB)
Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl (229 kB)
Using cached pytz-2026.3.post1-py2.py3-none-any.whl (508 kB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)
Using cached six-1.17.0-py2.py3-none-any.whl (11 kB)
  Attempting uninstall: pytz
    Found existing installation: pytz 2026.3.post1
    Uninstalling pytz-2026.3.post1:
      Successfully uninstalled pytz-2026.3.post1
  Attempt

In [2]:
import os
import warnings
import logging
from langchain_openai import ChatOpenAI, OpenAIEmbeddings 
from langchain_chroma import Chroma
import bs4
from langchain.agents import AgentState, create_agent
from langchain_docling import DoclingLoader
from langchain.messages import MessageLikeRepresentation
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.tools import tool

### Environment setup

For this step you will need to: 
- get a Langchain API Key (https://docs.langchain.com/oss/python/deepagents/rag)
- be added DHInfra project by Florian and get DHInfa API kez

In [8]:
import os
from openai import OpenAI

In [10]:
os.environ["LANGCHAIN_API_KEY"] = "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" # insert your own Langchain key
os.environ["LANGCHAIN_TRACING_V2"] = "false"  # <-- FIX 1: Disabled to prevent 403 error
os.environ["LANGCHAIN_PROJECT"] = "DHInfra-Tracing-Demo"
os.environ["LANGSMITH_DISABLE_RUN_COMPRESSION"] = "true"
os.environ["USER_AGENT"] = "my_agent"
os.environ["DHINFRA_API_KEY"] = "dhinfra_ef04f1a81a0a2388.GZhGekamhgsOMQoKgqq4jFW7bILScsv4dmd_HviJrcw" # insert DHInfra key

In [14]:
# <-- FIX 2: Custom class to prevent the 422 "null content" error
class SanitizedChatOpenAI(ChatOpenAI):
    def _get_request_payload(self, input_, *args, **kwargs):
        payload = super()._get_request_payload(input_, *args, **kwargs)
        if "messages" in payload:
            for msg in payload["messages"]:
                if msg.get("content") is None:
                    msg["content"] = ""
        return payload

# Initialize chat model using the sanitized class
model = SanitizedChatOpenAI(
    model="qwen3.5-397b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1",
    model_kwargs={"parallel_tool_calls": False}
)

# Initialize embedding model
embeddings = OpenAIEmbeddings(
    model="qwen3-embedding-8b",
    openai_api_key=os.environ["DHINFRA_API_KEY"],
    openai_api_base="https://api.dhinfra.uni-graz.at/v1"
)

# Initialize Chroma vector store
vector_store = Chroma(
    collection_name="rag_collection",
    embedding_function=embeddings
)

print("Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done")

Chat model (Qwen), embedding model (Qwen3-Embedding-8B), and Chroma vector store setup done


### this show us aann LLM response (without a specialized RAG) to the question "What is self-reflection?"

In [15]:
# Initialize client pointing to the DH-Infra API
client = OpenAI(
    api_key=os.environ.get("DHINFRA_API_KEY"),
    base_url="https://api.dhinfra.uni-graz.at/v1"
)

# Call the chat completion endpoint
response = client.chat.completions.create(
    model="qwen3.5-397b",
    messages=[
        {"role": "user", "content": "What is Self-Reflection?"}
    ]
)

# Print response content
print(response.choices[0].message.content)



**Self-reflection** is the practice of intentionally looking inward to examine your own thoughts, feelings, emotions, actions, and motivations. It is the ability to witness and evaluate your own cognitive and behavioral processes.

Think of it as a **mental debrief**. Just as a sports team watches game tape to see what worked and what didn't, self-reflection allows you to review your life experiences to learn from them.

Here is a breakdown of what self-reflection entails, why it matters, and how to do it effectively.

---

### 1. The Core Components
Self-reflection isn't just daydreaming or worrying. It involves three specific steps:
*   **Awareness:** Noticing what you are thinking or feeling in the moment (or after an event).
*   **Analysis:** Asking *why* you felt or acted that way. What triggered it? What values were involved?
*   **Adjustment:** Deciding how to apply that insight to future behavior.

### 2. Why Is It Important?
Without self-reflection, we tend to operate on "au

### Adding a custom RAG (in this case from a website)

In [16]:
import warnings
import logging
import os

# 1. Suppress general warnings FIRST (fixes the langchain-community warning)
warnings.filterwarnings("ignore")
# Optional: suppresses the Hugging Face token warningos.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1" 

# NOW we can import the rest safely
from langchain_community.vectorstores.utils import filter_complex_metadata
from transformers import logging as tf_logging

# 2. Suppress Docling logs & HF Transformers logs
logging.getLogger("docling").setLevel(logging.ERROR)
tf_logging.set_verbosity_error()

# Setup RAG: Load document using Docling
loader = DoclingLoader(file_path="https://lilianweng.github.io/posts/2023-06-23-agent/")
docs = loader.load()

# Split document into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
all_splits = text_splitter.split_documents(docs)

# Filter out complex nested metadata (dicts, lists) for ChromaDB compatibility
filtered_splits = filter_complex_metadata(all_splits) # there something may need to be adjusted depending on the data

# Index cleaned chunks in Chroma
_ = vector_store.add_documents(documents=filtered_splits)

print("Document loaded, split, and indexed successfully without warnings!")

Document loaded, split, and indexed successfully without warnings!


In [28]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

# 1. Redefine the retrieval tool returning a plain string
@tool
def retrieve_context(query: str) -> str:
    """Retrieve information to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    
    serialized = "\n\n".join(
        f"Source: {doc.metadata.get('source')}\nContent: {doc.page_content}"
        for doc in retrieved_docs
    )
    return serialized if serialized else "No relevant context found."

# 2. Create agent
tools = [retrieve_context]
system_prompt = (
    "You have access to a tool that retrieves context from a blog post. "
    "Use the tool to help answer user queries."
)

# Wir nutzen das sichere 'model', das du bereits in der ersten Zelle geladen hast!
agent = create_react_agent(
    model, 
    tools, 
    prompt=system_prompt
)

print("Agent created successfully!")

Agent created successfully!


Note: whatever metadata we want to have (eg sender, topic, sentiment) we need to adapt the document processing function for each individual use case

### What is self-reflection, in The context of agentic LLMs (answer with addes RAG)

In [22]:
query = "What is Self-Reflection?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is Self-Reflection?
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-b43cd29078edf8d9)
 Call ID: chatcmpl-tool-b43cd29078edf8d9
  Args:
    query: What is Self-Reflection?
================================= Tool Message =================================
Name: retrieve_context

Source: https://lilianweng.github.io/posts/2023-06-23-agent/
Content: Agent System Overview
In a LLM-powered autonomous agent system, LLM functions as the agent's brain, complemented by several key components:

Source: https://lilianweng.github.io/posts/2023-06-23-agent/
Content: - Long-term memory as the external vector store that the agent can attend to at query time, accessible via fast retrieval.
================================== Ai Message ==================================
Tool Calls:
  retrieve_context (chatcmpl-tool-beb98ad0c4a84770)
 Call ID:

In [31]:
for d in docs:
    print (d.metadata)

{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'dl_meta': {'schema_name': 'docling_core.transforms.chunker.DocMeta', 'version': '1.0.0', 'doc_items': [{'self_ref': '#/texts/9', 'parent': {'$ref': '#/texts/8'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}, {'self_ref': '#/texts/10', 'parent': {'$ref': '#/groups/2'}, 'children': [], 'content_layer': 'body', 'label': 'text', 'prov': []}, {'self_ref': '#/texts/11', 'parent': {'$ref': '#/groups/3'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/12', 'parent': {'$ref': '#/groups/3'}, 'children': [{'$ref': '#/groups/4'}], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/13', 'parent': {'$ref': '#/groups/4'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []}, {'self_ref': '#/texts/14', 'parent': {'$ref': '#/groups/4'}, 'children': [], 'content_layer': 'body', 'label': 'list_item', 'prov': []

In [26]:
query = "What is Self-Reflection?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": query}]},
    stream_mode="updates",
):
   
    print(step)

{'agent': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 77, 'prompt_tokens': 299, 'total_tokens': 376, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3.5-397b', 'system_fingerprint': 'vllm-0.27.1-tp4-ep-85d4168a', 'id': 'chatcmpl-bb27de6790bf661b', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a019d6-ff45-7551-96fb-97d1f1c4941a-0', tool_calls=[{'name': 'retrieve_context', 'args': {'query': 'Self-Reflection'}, 'id': 'chatcmpl-tool-a55286b9de88aa88', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 299, 'output_tokens': 77, 'total_tokens': 376, 'input_token_details': {}, 'output_token_details': {}})]}}
{'tools': {'messages': [ToolMessage(content='Source: https://lilianweng.github.io/posts/2023-06-23-agent/\nContent: **MRKL**\n(\n[Karpas et al. 2022](https://arxiv.org/abs/2205.00445)\n), short f

## Adapting the code for our own data

If it is a dataframe:

In [32]:
import warnings
import logging
import os
import pandas as pd

warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores.utils import filter_complex_metadata


In [33]:
import pandas as pd
from langchain_core.documents import Document

### Load local CSV

In [34]:
df = pd.read_csv("MigraAnno.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")

In [35]:
print(df.columns)


Index(['id', 'text', 'Topic', 'Name-original', 'newspaper_title', 'date',
       'preceding_document', 'following_document', 'Relevancy_proba',
       'sentiment', 'year', 'Category'],
      dtype='object')


### Convert CSV rows into LangChain Documents

In [63]:
def clean_value(value):
    return "" if pd.isna(value) else value


docs = []

for _, row in df.iterrows():

    # Text used for semantic retrieval - this part is squished into a string together with the text, 
    #so it works well with querying, the rest of the metadata is (for now) disregarded
    page_content = f"""
Topic: {clean_value(row['Name-original'])}
Sentiment: {clean_value(row['sentiment'])}
Category: {clean_value(row['Category'])}
Year: {clean_value(row['year'])}


{clean_value(row['text'])}[''
""".strip()

    # Structured information stored alongside each document
    metadata = {
        "id": clean_value(row["id"]),
        "topic": clean_value(row["Topic"]),
        "name_original": clean_value(row["Name-original"]),
        "newspaper_title": clean_value(row["newspaper_title"]),
        "date": clean_value(row["date"]),
        "preceding_document": clean_value(row["preceding_document"]),
        "following_document": clean_value(row["following_document"]),
        "relevancy_proba": clean_value(row["Relevancy_proba"]),
        "sentiment": clean_value(row["sentiment"]),
        "year": clean_value(row["year"]),
        "category": clean_value(row["Category"]),
    }

    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata
        )
    )

A subset of the corpus

In [64]:
import random

print("Filtering metadata...")
filtered_docs = filter_complex_metadata(docs)

filtered_docs = random.sample(
    filtered_docs,
    min(10000, len(filtered_docs))
)

print(f"Filtering done: {len(filtered_docs)} documents")

print("Starting vector store indexing...")
batch_size = 100

for start in range(0, len(filtered_docs), batch_size):
    end = min(start + batch_size, len(filtered_docs))

    vector_store.add_documents(
        documents=filtered_docs[start:end]
    )

    print(f"Indexed {end}/{len(filtered_docs)} documents")

Filtering metadata...
Filtering done: 10000 documents
Starting vector store indexing...
Indexed 100/10000 documents
Indexed 200/10000 documents
Indexed 300/10000 documents
Indexed 400/10000 documents
Indexed 500/10000 documents
Indexed 600/10000 documents
Indexed 700/10000 documents
Indexed 800/10000 documents
Indexed 900/10000 documents
Indexed 1000/10000 documents
Indexed 1100/10000 documents
Indexed 1200/10000 documents
Indexed 1300/10000 documents
Indexed 1400/10000 documents
Indexed 1500/10000 documents
Indexed 1600/10000 documents
Indexed 1700/10000 documents
Indexed 1800/10000 documents
Indexed 1900/10000 documents
Indexed 2000/10000 documents
Indexed 2100/10000 documents
Indexed 2200/10000 documents
Indexed 2300/10000 documents
Indexed 2400/10000 documents
Indexed 2500/10000 documents
Indexed 2600/10000 documents
Indexed 2700/10000 documents
Indexed 2800/10000 documents
Indexed 2900/10000 documents
Indexed 3000/10000 documents
Indexed 3100/10000 documents
Indexed 3200/10000 doc

The whole corpus

In [37]:
print("Filtering metadata...")
filtered_docs = filter_complex_metadata(docs)
print(f"Filtering done: {len(filtered_docs)} documents")

print("Starting vector store indexing...")
batch_size = 100

for start in range(0, len(filtered_docs), batch_size):
    end = min(start + batch_size, len(filtered_docs))

    vector_store.add_documents(
        documents=filtered_docs[start:end]
    )

    print(f"Indexed {end}/{len(filtered_docs)} documents")

Filtering metadata...
Filtering done: 96871 documents
Starting vector store indexing...
Indexed 100/96871 documents
Indexed 200/96871 documents
Indexed 300/96871 documents
Indexed 400/96871 documents
Indexed 500/96871 documents
Indexed 600/96871 documents
Indexed 700/96871 documents
Indexed 800/96871 documents
Indexed 900/96871 documents
Indexed 1000/96871 documents
Indexed 1100/96871 documents
Indexed 1200/96871 documents
Indexed 1300/96871 documents
Indexed 1400/96871 documents
Indexed 1500/96871 documents
Indexed 1600/96871 documents
Indexed 1700/96871 documents
Indexed 1800/96871 documents
Indexed 1900/96871 documents
Indexed 2000/96871 documents
Indexed 2100/96871 documents
Indexed 2200/96871 documents
Indexed 2300/96871 documents
Indexed 2400/96871 documents
Indexed 2500/96871 documents
Indexed 2600/96871 documents
Indexed 2700/96871 documents
Indexed 2800/96871 documents
Indexed 2900/96871 documents
Indexed 3000/96871 documents
Indexed 3100/96871 documents
Indexed 3200/96871 doc

KeyboardInterrupt: 

In [71]:
results = vector_store.similarity_search(
    "Negative sentiment documents in year 1839",
    k=10
)

In [72]:
results

[Document(id='54646deb-b5f7-4348-9d73-83221753ddea', metadata={'year': 1830, 'topic': 20, 'newspaper_title': 'obo', 'id': 43999.0, 'relevancy_proba': 0.99995184, 'name_original': '20_zug_passagier_bahn_eisenbahn', 'following_document': 'Die altesten hatten acht und dreißig Jahre Freud und Leid in Lieb, und Eintracht mit ihrem Herrn getheilt, die jüngern waren alle unter einer Herrschaft herangewachsen, waren gewohnt, einen Herrn, Vater zu nennen, und erhielten eher, durch diese Feierlichkeit die Versicherung, daß in einer Gott gebe, sehr fernen Zeit, Sie den Vater nicht verlieren, sondern nur wechseln würden. Die Krönung ging nach der vorgeschriebenen Weise vorsich.', 'category': 'CTX', 'sentiment': 'neutral', 'date': '1830-10-01', 'preceding_document': 'Vier Herold, ein neuer Hofstaat steigerten die gesammte Aufmerksamkeit, und endlich nahten Ihre Majestäten im Imperial=Ballawagen, von acht Schimmeln gezogen. Es war ein wahrhaft kaiserlicher Anblick!'}, page_content="Topic: 20_zug_pas

### If it is an XML (just a skeleton code)

More information:

LangChain provides an `UnstructuredXMLLoader` for converting XML files into LangChain `Document` objects. [[1]](#ref1)

For richly structured XML, particularly **TEI XML**, it can be advantageous to exploit the encoded document structure rather than treating the XML as undifferentiated text. Castellon, Chiffoleau, and Miasnikova demonstrate how XML-TEI digital editions can serve as the knowledge source for a RAG system. [[2]](#ref2)


### References

<a id="ref1"></a>
**[1]** LangChain. *UnstructuredXMLLoader*. `langchain_community.document_loaders`.  
https://reference.langchain.com/python/langchain-community/document_loaders/xml/UnstructuredXMLLoader

<a id="ref2"></a>
**[2]** Castellon, C., Chiffoleau, F., & Miasnikova, A. (2026). *Faire du neuf avec du balisé : Quand une édition TEI devient la mémoire d'un RAG*. *Anthology of Computers and the Humanities*, 4.  
https://anthology.ach.org/volumes/vol0004/faire-du-neuf-avec-du-balis-quand-une-dition-tei-devient-la/

In [ ]:
from pathlib import Path
from lxml import etree

from langchain_core.documents import Document
from langchain_community.vectorstores.utils import filter_complex_metadata


In [ ]:
TEI_NS = {"tei": "http://www.tei-c.org/ns/1.0"}

xml_folder = Path("data/letters")

docs = []


In [ ]:
for xml_file in xml_folder.glob("*.xml"):

    tree = etree.parse(str(xml_file))
    root = tree.getroot()

   
    # Extract correspondence metadata
 

    letter_id = root.get(
        "{http://www.w3.org/XML/1998/namespace}id",
        xml_file.stem
    )

    sender = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:persName)",
        namespaces=TEI_NS
    ).strip()

    recipient = root.xpath(
        "string(.//tei:correspAction[@type='received']/tei:persName)",
        namespaces=TEI_NS
    ).strip()

    date = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:date/@when)",
        namespaces=TEI_NS
    ).strip()

    sender_place = root.xpath(
        "string(.//tei:correspAction[@type='sent']/tei:placeName)",
        namespaces=TEI_NS
    ).strip()

    recipient_place = root.xpath(
        "string(.//tei:correspAction[@type='received']/tei:placeName)",
        namespaces=TEI_NS
    ).strip()

    # Extract letter text

    body_nodes = root.xpath(
        ".//tei:text/tei:body",
        namespaces=TEI_NS
    )

    if body_nodes:
        body_text = " ".join(
            " ".join(body_nodes[0].itertext()).split()
        )
    else:
        body_text = ""

   
    # Construct text for embeddings
   
    page_content = f"""
Sender: {sender}
Recipient: {recipient}
Date: {date}

{body_text}
""".strip()

 
    # Metadata
    
    metadata = {
        "id": letter_id,
        "sender": sender,
        "recipient": recipient,
        "date": date,
        "sender_place": sender_place,
        "recipient_place": recipient_place,
        "source_file": xml_file.name,
    }

    docs.append(
        Document(
            page_content=page_content,
            metadata=metadata
        )
    )

print(f"{len(docs)} letters loaded.")

In [ ]:
print(docs[0].page_content)

print("\nMETADATA:")
print(docs[0].metadata)

In [ ]:
filtered_docs = filter_complex_metadata(docs)

print(f"Indexing {len(filtered_docs)} letters...")

vector_store.add_documents(
    documents=filtered_docs
)

print("Indexing complete.")

### Test semantic retreival

In [ ]:
query = "What do the correspondents say about linguistics?"

results = vector_store.similarity_search(
    query,
    k=5
)

for i, doc in enumerate(results, start=1):
    
    print(f"\n--- Result {i} ---")
    print("Sender:", doc.metadata["sender"])
    print("Recipient:", doc.metadata["recipient"])
    print("Date:", doc.metadata["date"])
    print("Source:", doc.metadata["source_file"])
    
    print()
    print(doc.page_content[:500])

## We now worked on an embedding-based RAG. What  other types of RAGs exist? 

### Graph-based RAG

**Graph-based RAG (Graph RAG)** approaches retrieval from a different perspective. Instead of representing the collection primarily as independent vectors, information is represented through **entities (nodes) and relationships (edges)**.[^3]

For example, a newspaper corpus could contain relationships such as:

```text
Newspaper
    │
 published
    ▼
 Document ── has_topic ──→ Topic
    │
    ├──── mentions ──────→ Person
    │
    ├── has_sentiment ───→ Negative
    │
    └──── belongs_to ────→ Category
```

Retrieval can therefore follow relationships between entities rather than relying exclusively on semantic similarity.

This is particularly useful for questions involving several interconnected properties, for example:

> *Which topics were associated with negatively represented people in a particular newspaper during the 1930s?*

Such a question may require traversal through several relationships:

```text
Newspaper
    ↓
Documents
    ↓
People
    ↓
Sentiment
    ↓
Topics
```

Graph-based retrieval is therefore particularly useful when the research question concerns **how entities are connected**, rather than simply which documents are semantically similar.

### Embedding-based RAG vs. Graph RAG

The fundamental difference is therefore the type of information used to determine relevance:

|                          | Embedding-based RAG      | Graph-based RAG                        |
| ------------------------ | ------------------------ | -------------------------------------- |
| Representation           | Numerical vectors        | Nodes and relationships                |
| Retrieval                | Semantic similarity      | Graph relationships/traversal          |
| Typical question         | *Which texts discuss X?* | *How is X related to Y?*               |
| Unstructured text        | Very suitable            | Usually requires additional modelling  |
| Explicit relationships   | Weak                     | Strong                                 |
| Metadata filtering       | Yes                      | Relationships can be modelled directly |
| Complexity               | Relatively low           | Higher                                 |
| Multi-step relationships | Limited                  | Particularly suitable                  |

These approaches are not mutually exclusive. **Hybrid RAG architectures can combine embedding similarity with graph traversal.** For example, LangChain's graph retriever can begin with vector similarity and subsequently traverse relationships defined through document metadata.[^3]

Conceptually:

```text
Question → Embedding search → Relevant documents → Graph relationships → Related documents → LLM|
```

For some corpora, **embedding-based RAG can be a good option**. 

In the case of MigraAnno, The document `text` (newspaper article) can be embedded for semantic retrieval, while fields such as `Topic`, `newspaper_title`, `date`, `sentiment`, `year`, and `Category` can be retained as structured metadata.

The `preceding_document` and `following_document` fields additionally encode explicit relationships between documents. They could therefore later provide a natural basis for graph-based or hybrid retrieval.

In short:

> **Embedding-based RAG retrieves information because it is semantically similar; Graph RAG retrieves information because it is explicitly connected.**

A hybrid system can exploit both forms of evidence.



### References

<a id="ref1"></a>
**[1]** LangChain. *Retrieval-Augmented Generation (RAG) with Deep Agents*. LangChain Documentation.  
https://docs.langchain.com/oss/python/deepagents/rag

<a id="ref2"></a>
**[2]** Chroma. *Chroma Documentation*.  
https://docs.trychroma.com/

<a id="ref3"></a>
**[3]** LangChain. *Graph RAG / GraphRetriever*. LangChain Documentation.  
https://docs.langchain.com/oss/python/integrations/retrievers/graph_rag